In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")

print("Path to dataset files:", path)




Using Colab cache for faster access to the 'garbage-classification' dataset.
Path to dataset files: /kaggle/input/garbage-classification


In [5]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.3 MB/s eta 0:00:00


In [9]:
import os
import shutil
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

# Ensure ultralytics is installed
!pip install ultralytics -q

# Dataset path from the previous cell, initially points to the kagglehub download root.
initial_dataset_path = path
print(f"Initial dataset path: {initial_dataset_path}")

# Adjust dataset_path to point to the actual root containing image class folders
true_dataset_root = None
for first_level_dir in os.listdir(initial_dataset_path):
    first_level_path = os.path.join(initial_dataset_path, first_level_dir)
    if os.path.isdir(first_level_path):
        # Look for a common sub-directory, often named after the dataset itself
        for second_level_dir in os.listdir(first_level_path):
            if second_level_dir.lower() == 'garbage classification': # Assuming this is the common nested directory name
                potential_root = os.path.join(first_level_path, second_level_dir)
                # Check if this potential_root actually contains subdirectories (which would be our classes)
                if os.path.isdir(potential_root) and len([d for d in os.listdir(potential_root) if os.path.isdir(os.path.join(potential_root, d))]) > 0:
                    true_dataset_root = potential_root
                    break
        if true_dataset_root:
            break

if not true_dataset_root:
    raise ValueError("Could not determine the correct dataset root containing image class directories. "
                     "Please inspect the dataset structure manually.")

dataset_path = true_dataset_root
print(f"Adjusted dataset path to actual image classes root: {dataset_path}")

# Define new base directory for processed dataset
base_data_dir = '/content/garbage_classification_split'

# Clean up previous runs if any
if os.path.exists(base_data_dir):
    shutil.rmtree(base_data_dir)
os.makedirs(base_data_dir, exist_ok=True)

# Define train, val, test directories
train_dir = os.path.join(base_data_dir, 'train')
val_dir = os.path.join(base_data_dir, 'val')
test_dir = os.path.join(base_data_dir, 'test')

# Get class names by inspecting the dataset_path's subdirectories
class_names = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
if not class_names:
    raise ValueError("No class subdirectories found in the adjusted dataset path. \n"\
                       "Please ensure your dataset is structured as: dataset_path/class_name/image.jpg")
print(f"Found classes: {class_names}")

# Create class subdirectories in train, val, test
for cls_name in class_names:
    os.makedirs(os.path.join(train_dir, cls_name), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls_name), exist_ok=True)

# Split data and copy to new directories with stratification
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

print("Splitting dataset into train, validation, and test sets with stratification...")
all_images = []
all_labels = []

for cls_idx, cls_name in enumerate(class_names):
    class_path = os.path.join(dataset_path, cls_name)
    images_in_class = [os.path.join(class_path, img) for img in os.listdir(class_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
    all_images.extend(images_in_class)
    all_labels.extend([cls_idx] * len(images_in_class)) # Use class index for stratification

# Add a check for empty all_images here to give a more specific error if needed.
if not all_images:
    raise ValueError("No image files found in any class subdirectories. Please check your dataset structure and file extensions.")

# Perform initial split into train and (val+test)
train_images, temp_images, train_labels, temp_labels = train_test_split(
    all_images, all_labels, test_size=(val_ratio + test_ratio), random_state=42, stratify=all_labels
)

# Perform split of (val+test) into val and test
val_images, test_images, val_labels, test_labels = train_test_split(
    temp_images, temp_labels, test_size=test_ratio / (val_ratio + test_ratio), random_state=42, stratify=temp_labels
)

# Function to copy files
def copy_files(image_list, dest_base_dir):
    for img_path in image_list:
        # Determine class name from image path (parent directory of the image file)
        parent_dir = os.path.basename(os.path.dirname(img_path))
        shutil.copy(img_path, os.path.join(dest_base_dir, parent_dir, os.path.basename(img_path)))

print("Copying files to train directory...")
copy_files(train_images, train_dir)
print("Copying files to validation directory...")
copy_files(val_images, val_dir)
print("Copying files to test directory...")
copy_files(test_images, test_dir)

print("Dataset splitting and copying complete.")
print(f"Train samples: {len(train_images)}")
print(f"Validation samples: {len(val_images)}")
print(f"Test samples: {len(test_images)}")

# Create data.yaml for YOLOv8 (This file is not directly used for classification training, but can be informative)
data_yaml_path = os.path.join(base_data_dir, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(f"path: {base_data_dir}\n")
    f.write(f"train: train\n")
    f.write(f"val: val\n")
    f.write(f"test: test\n")
    f.write(f"nc: {len(class_names)}\n")
    f.write(f"names: {class_names}\n")

print(f"data.yaml created at: {data_yaml_path}")

# Initialize a YOLO classification model for transfer learning
# Using 'yolov8n-cls.pt' for a nano classification model suitable for Raspberry Pi
model = YOLO('yolov8n-cls.pt')

# Train the model
print("Starting model training...")
# Adjust epochs, imgsz, and patience based on your needs and available resources.
# patience=5 will stop training if validation loss doesn't improve for 5 epochs.
results = model.train(data=base_data_dir, epochs=20, imgsz=128, project='garbage_classification_yolov8n', name='yolov8n_cls_transfer', patience=5)
print("Model training complete.")

# Evaluate the model on the test set
print("Starting model evaluation on the test set...")
# For classification, the 'data' argument should be the root directory, and 'split' indicates which split to use.
metrics = model.val(data=base_data_dir, split='test')
print("Model evaluation complete.")

# Print key evaluation metrics
print("\nEvaluation Metrics on Test Set:")
print(f"Top-1 Accuracy: {metrics.top1:.4f}")
print(f"Top-5 Accuracy: {metrics.top5:.4f}")

# Optional: Export the model for deployment on Raspberry Pi (e.g., to ONNX format)
# print("\nExporting model to ONNX format...")
# model.export(format='onnx', filename='yolov8n_garbage_classifier.onnx')
# print("Model exported to yolov8n_garbage_classifier.onnx")

Initial dataset path: /kaggle/input/garbage-classification
Adjusted dataset path to actual image classes root: /kaggle/input/garbage-classification/Garbage classification/Garbage classification
Found classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Splitting dataset into train, validation, and test sets with stratification...
Copying files to train directory...
Copying files to validation directory...
Copying files to test directory...
Dataset splitting and copying complete.
Train samples: 1768
Validation samples: 379
Test samples: 380
data.yaml created at: /content/garbage_classification_split/data.yaml
Starting model training...
Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mo

In [10]:
import shutil
import os

source_dir = "/content/runs/classify/garbage_classification_yolov8n/yolov8n_cls_transfer2"
output_filename = "yolov8n_cls_transfer2_results"

# Create a zip archive of the directory
shutil.make_archive(output_filename, 'zip', source_dir)

print(f"Successfully created {output_filename}.zip in /content/")

Successfully created yolov8n_cls_transfer2_results.zip in /content/


In [12]:
# Dataset path from the previous cell, initially points to the kagglehub download root.


path = kagglehub.dataset_download("mostafaabla/garbage-classification")
print("Path to dataset files:", path)

initial_dataset_path = path
print(f"Initial dataset path: {initial_dataset_path}")


Using Colab cache for faster access to the 'garbage-classification' dataset.
Path to dataset files: /kaggle/input/garbage-classification
Initial dataset path: /kaggle/input/garbage-classification


In [15]:
# Adjust dataset_path to point to the actual root containing image class folders
true_dataset_root = None

# Iterate through first level subdirectories of the initial dataset path
for first_level_dir in os.listdir(initial_dataset_path):
    first_level_path = os.path.join(initial_dataset_path, first_level_dir)
    if os.path.isdir(first_level_path):
        # Check if this first_level_path directly contains class subdirectories (e.g., 'cardboard', 'glass')
        subdirs = [d for d in os.listdir(first_level_path) if os.path.isdir(os.path.join(first_level_path, d))]
        # Heuristic: if there are multiple subdirectories and at least one contains image files
        if len(subdirs) > 1 and \
           any(any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in os.listdir(os.path.join(first_level_path, sd))) for sd in subdirs):
            true_dataset_root = first_level_path
            break # Found it, exit loops

        # Original logic: Look for a common sub-directory, often named after the dataset itself, e.g., 'Garbage classification'
        for second_level_dir in os.listdir(first_level_path):
            if second_level_dir.lower() == 'garbage classification':
                potential_root = os.path.join(first_level_path, second_level_dir)
                # Check if this potential_root actually contains subdirectories (which would be our classes)
                if os.path.isdir(potential_root) and len([d for d in os.listdir(potential_root) if os.path.isdir(os.path.join(potential_root, d))]) > 0:
                    # Additional check if these subdirectories contain images
                    class_subdirs = [d for d in os.listdir(potential_root) if os.path.isdir(os.path.join(potential_root, d))]
                    if any(any(f.lower().endswith(('.png', '.jpg', '.jpeg')) for f in os.listdir(os.path.join(potential_root, csd))) for csd in class_subdirs):
                        true_dataset_root = potential_root
                        break
        if true_dataset_root:
            break

if not true_dataset_root:
    raise ValueError("Could not determine the correct dataset root containing image class directories. "
                     "Please inspect the dataset structure manually.")

dataset_path = true_dataset_root
print(f"Adjusted dataset path to actual image classes root: {dataset_path}")

# Define new base directory for processed dataset
base_data_dir = '/content/garbage_classification_split'

# Clean up previous runs if any
if os.path.exists(base_data_dir):
    shutil.rmtree(base_data_dir)
os.makedirs(base_data_dir, exist_ok=True)

# Define train, val, test directories
train_dir = os.path.join(base_data_dir, 'train')
val_dir = os.path.join(base_data_dir, 'val')
test_dir = os.path.join(base_data_dir, 'test')

# Get class names by inspecting the dataset_path's subdirectories
class_names = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
if not class_names:
    raise ValueError("No class subdirectories found in the adjusted dataset path. \n"\
                       "Please ensure your dataset is structured as: dataset_path/class_name/image.jpg")
print(f"Found classes: {class_names}")

# Create class subdirectories in train, val, test
for cls_name in class_names:
    os.makedirs(os.path.join(train_dir, cls_name), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls_name), exist_ok=True)

# Split data and copy to new directories with stratification
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

print("Splitting dataset into train, validation, and test sets with stratification...")
all_images = []
all_labels = []

for cls_idx, cls_name in enumerate(class_names):
    class_path = os.path.join(dataset_path, cls_name)
    images_in_class = [os.path.join(class_path, img) for img in os.listdir(class_path) if img.lower().endswith(('.png', '.jpg', '.jpeg'))]
    all_images.extend(images_in_class)
    all_labels.extend([cls_idx] * len(images_in_class)) # Use class index for stratification

# Add a check for empty all_images here to give a more specific error if needed.
if not all_images:
    raise ValueError("No image files found in any class subdirectories. Please check your dataset structure and file extensions.")

# Perform initial split into train and (val+test)
train_images, temp_images, train_labels, temp_labels = train_test_split(
    all_images, all_labels, test_size=(val_ratio + test_ratio), random_state=42, stratify=all_labels
)

# Perform split of (val+test) into val and test
val_images, test_images, val_labels, test_labels = train_test_split(
    temp_images, temp_labels, test_size=test_ratio / (val_ratio + test_ratio), random_state=42, stratify=temp_labels
)

# Function to copy files
def copy_files(image_list, dest_base_dir):
    for img_path in image_list:
        # Determine class name from image path (parent directory of the image file)
        parent_dir = os.path.basename(os.path.dirname(img_path))
        shutil.copy(img_path, os.path.join(dest_base_dir, parent_dir, os.path.basename(img_path)))

print("Copying files to train directory...")
copy_files(train_images, train_dir)
print("Copying files to validation directory...")
copy_files(val_images, val_dir)
print("Copying files to test directory...")
copy_files(test_images, test_dir)

print("Dataset splitting and copying complete.")
print(f"Train samples: {len(train_images)}")
print(f"Validation samples: {len(val_images)}")
print(f"Test samples: {len(test_images)}")

# Create data.yaml for YOLOv8 (This file is not directly used for classification training, but can be informative)
data_yaml_path = os.path.join(base_data_dir, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(f"path: {base_data_dir}\n")
    f.write(f"train: train\n")
    f.write(f"val: val\n")
    f.write(f"test: test\n")
    f.write(f"nc: {len(class_names)}\n")
    f.write(f"names: {class_names}\n")

print(f"data.yaml created at: {data_yaml_path}")

# Initialize a YOLO classification model for transfer learning
# Using 'yolov8n-cls.pt' for a nano classification model suitable for Raspberry Pi
model = YOLO('yolov8n-cls.pt')

# Train the model
print("Starting model training...")
# Adjust epochs, imgsz, and patience based on your needs and available resources.
# patience=5 will stop training if validation loss doesn't improve for 5 epochs.
results = model.train(data=base_data_dir, epochs=20, imgsz=128, project='garbage_classification2_yolov8n', name='yolov8n_cls_transfer', patience=5)
print("Model training complete.")

# Evaluate the model on the test set
print("Starting model evaluation on the test set...")
# For classification, the 'data' argument should be the root directory, and 'split' indicates which split to use.
metrics = model.val(data=base_data_dir, split='test')
print("Model evaluation complete.")

# Print key evaluation metrics
print("\nEvaluation Metrics on Test Set:")
print(f"Top-1 Accuracy: {metrics.top1:.4f}")
print(f"Top-5 Accuracy: {metrics.top5:.4f}")

# Optional: Export the model for deployment on Raspberry Pi (e.g., to ONNX format)
# print("\nExporting model to ONNX format...")
# model.export(format='onnx', filename='yolov8n_garbage_classifier.onnx')
# print("Model exported to yolov8n_garbage_classifier.onnx")

Adjusted dataset path to actual image classes root: /kaggle/input/garbage-classification/garbage_classification
Found classes: ['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']
Splitting dataset into train, validation, and test sets with stratification...
Copying files to train directory...
Copying files to validation directory...
Copying files to test directory...
Dataset splitting and copying complete.
Train samples: 10860
Validation samples: 2327
Test samples: 2328
data.yaml created at: /content/garbage_classification_split/data.yaml
Starting model training...
Ultralytics 8.4.38 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_

In [ ]:
import os
import shutil
from ultralytics import YOLO

# 1. Define paths and parameters
# Path to the best weights from the previous training run
model_path = '/content/runs/classify/garbage_classification2_yolov8n/yolov8n_cls_transfer/weights/best.pt'
original_data_base_dir = '/content/garbage_classification_split' # This variable is available from previous cells
binary_data_dir = '/content/garbage_binary_classification'

# Class names from the original training, available in kernel state
class_names_original = ['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']
trash_class_name = 'trash'

binary_class_names = ['not_trash', 'trash']

# Clean up previous binary data directory if it exists
if os.path.exists(binary_data_dir):
    shutil.rmtree(binary_data_dir)
os.makedirs(binary_data_dir, exist_ok=True)

# 2. Create new binary dataset structure
splits = ['train', 'val', 'test']
for split in splits:
    os.makedirs(os.path.join(binary_data_dir, split, 'trash'), exist_ok=True)
    os.makedirs(os.path.join(binary_data_dir, split, 'not_trash'), exist_ok=True)

# 3. Populate binary dataset
print(f"Creating binary dataset at {binary_data_dir}...")
for split in splits:
    original_split_path = os.path.join(original_data_base_dir, split)
    for class_name in class_names_original:
        source_class_path = os.path.join(original_split_path, class_name)
        if os.path.exists(source_class_path):
            for img_name in os.listdir(source_class_path):
                source_img_path = os.path.join(source_class_path, img_name)
                if class_name == trash_class_name:
                    dest_path = os.path.join(binary_data_dir, split, 'trash', img_name)
                else:
                    dest_path = os.path.join(binary_data_dir, split, 'not_trash', img_name)
                shutil.copy(source_img_path, dest_path)
print("Binary dataset created.")

# 4. Create data.yaml for YOLOv8 binary classification
data_yaml_binary_path = os.path.join(binary_data_dir, 'data.yaml')
with open(data_yaml_binary_path, 'w') as f:
    f.write(f"path: {binary_data_dir}\n")
    f.write(f"train: train\n")
    f.write(f"val: val\n")
    f.write(f"test: test\n")
    f.write(f"nc: {len(binary_class_names)}\n")
    f.write(f"names: {binary_class_names}\n")
print(f"Binary data.yaml created at: {data_yaml_binary_path}")

# 5. Load the pre-trained model
print(f"Loading pre-trained model from {model_path}...")
model = YOLO(model_path)
print("Model loaded successfully.")

# 6. Retrain (transfer learn) the model on the binary dataset
print("Starting model fine-tuning for binary classification (trash vs not_trash)...")
# Adjust epochs, imgsz, and patience as needed.
# For binary classification, typically fewer epochs might be needed if the base model is already good.
results = model.train(
    data=binary_data_dir,
    epochs=10,  # Reduced epochs for fine-tuning a binary task
    imgsz=128,  # Keep image size consistent or adjust as needed
    project='garbage_binary_classification_yolov8n',
    name='yolov8n_binary_transfer',
    patience=3 # Early stopping patience
)
print("Model fine-tuning complete.")

# 7. Evaluate the fine-tuned model on the test set
print("Starting model evaluation on the binary test set...")
metrics = model.val(data=binary_data_dir, split='test')
print("Model evaluation complete.")

# Print key evaluation metrics
print("\nEvaluation Metrics on Binary Test Set:")
print(f"Top-1 Accuracy: {metrics.top1:.4f}")
print(f"Top-5 Accuracy: {metrics.top5:.4f}") # Top-5 might not be meaningful for binary classification, but YOLO always reports it.

# Optional: Export the fine-tuned model
# print("\nExporting fine-tuned model to ONNX format...")
# model.export(format='onnx', filename='yolov8n_binary_trash_classifier.onnx')
# print("Fine-tuned model exported to yolov8n_binary_trash_classifier.onnx")


### Exporting the Fine-Tuned Model

The fine-tuning process performed in the previous cell effectively transfers knowledge from a large general dataset to your specific binary classification task, resulting in a specialized and efficient model suitable for deployment on edge devices like Raspberry Pi.

To prepare the model for deployment, it's common to export it to a lightweight format such as ONNX. This will create a `.onnx` file that can be used with various inference engines on your Raspberry Pi.

In [ ]:
# Load the best fine-tuned 12-class model
# This model was trained in cell 9mgmtv6jzzTX on the 'mostafaabla/garbage-classification' dataset.
model = YOLO('/content/runs/classify/garbage_classification2_yolov8n/yolov8n_cls_transfer/weights/best.pt')

# Export the fine-tuned model to ONNX format for Raspberry Pi deployment
print("\nExporting 12-class fine-tuned model to ONNX format...")
model.export(format='onnx', filename='yolov8n_12class_garbage_classifier.onnx')
print("12-class fine-tuned model exported to yolov8n_12class_garbage_classifier.onnx")